# 02 - Model Sanity Check
Verify model shapes, parameter count, and forward pass.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from model.chess_net import ChessNet
from model.board_encoder import BoardEncoder
from model.policy_head import get_legal_move_mask, move_to_policy_index, policy_index_to_move
import chess

## Model Parameters

In [ ]:
model = ChessNet()
print(f'Total parameters: {model.count_parameters():,}')

## Forward Pass with Random Input

In [ ]:
batch_size = 4
board = torch.randn(batch_size, 18, 8, 8)
elo = torch.tensor([1000.0, 1500.0, 2000.0, 2500.0])
mask = torch.ones(batch_size, 4672)

policy, value = model(board, elo, mask)
print(f'Policy shape: {policy.shape}')  # (4, 4672)
print(f'Value shape: {value.shape}')    # (4, 1)
print(f'Policy sum (should be ~1): {torch.exp(policy[0]).sum():.4f}')
print(f'Value range: [{value.min():.3f}, {value.max():.3f}]')

## Real Position Test

In [ ]:
encoder = BoardEncoder()
fen = 'rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1'
board = chess.Board(fen)

board_tensor = torch.from_numpy(encoder.encode_board(board)).unsqueeze(0)
elo_tensor = torch.tensor([1500.0])
legal_mask = torch.from_numpy(get_legal_move_mask(board)).unsqueeze(0)

policy, value = model(board_tensor, elo_tensor, legal_mask)
probs = torch.exp(policy[0])
top5 = probs.topk(5)

print(f'Top 5 moves after 1.e4:')
for prob, idx in zip(top5.values, top5.indices):
    move = policy_index_to_move(idx.item(), board)
    san = board.san(move) if move and move in board.legal_moves else '???'
    print(f'  {san}: {prob:.3f}')

## Move Encoding Round-Trip Test

In [ ]:
# Verify all legal moves can be encoded and decoded
board = chess.Board()
errors = 0
for move in board.legal_moves:
    idx = move_to_policy_index(move, flip=False)
    recovered = policy_index_to_move(idx, board)
    if recovered != move:
        print(f'MISMATCH: {move} -> idx {idx} -> {recovered}')
        errors += 1
print(f'Tested {len(list(board.legal_moves))} moves, {errors} errors')